# Fase 3 — Limpieza individual de los documentos aceptados

*Hábitos musicales, estado de ánimo, estrés académico y uso de IA en estudiantes de MAC, FES Acatlán*

En esta fase se depura **cada documento por separado**; la integración corresponde a la Fase 4. Para cada uno se diagnostica la estructura, se normaliza el texto, se convierten las respuestas abiertas en variables numéricas o categóricas con reglas explícitas, se tratan los faltantes, se detectan valores atípicos y se guarda el resultado en `data/processed/` con el sufijo `_clean`.

**Principios de la limpieza:**

1. **Reglas explícitas y auditables.** Cada variable se codifica con una lista ordenada de expresiones regulares, definida en `src/codificacion.py`. La primera que coincide determina el valor.
2. **No inventar datos.** Si una respuesta no permite asignar un valor sin suponer información que el estudiante no dio, se registra como faltante. No se imputa ningún valor.
3. **Conservar antes que eliminar.** Los atípicos y las respuestas sospechosas se **marcan**, no se borran. Así, la decisión de excluirlos se toma de forma explícita en el análisis.
4. **Anonimato.** El número de cuenta se sustituye por un seudónimo irreversible. Este cuaderno no muestra respuestas individuales ni identificadores.

Antes de redactar las reglas se revisaron, fuera de este cuaderno, todas las respuestas de cada variable junto a su código asignado. Así se ajustaron las reglas hasta que cada código coincidiera con el sentido literal de la respuesta.

In [ ]:
# Configuración
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

np.random.seed(42)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from datos import PROCESSED, RAW, cargar
from limpieza import a_horas, clave_seudonimo, normalizar
from codificacion import (CARACTERISTICAS, FASES, HERRAMIENTAS, MOMENTOS, PLATAFORMAS,
                          corregir_cuenta_invalida, limpiar_doc1, limpiar_doc2, limpiar_doc3)

pd.set_option("display.max_columns", 40)
crudos = {d: cargar(d) for d in (1, 2, 3)}


def resumen_codificacion(df, variables):
    """Valores válidos, faltantes y distribución de cada variable codificada (sin datos individuales)."""
    filas = []
    for v in variables:
        s = df[v]
        dist = s.value_counts(dropna=True).sort_index()
        filas.append({"variable": v, "válidos": int(s.notna().sum()), "faltantes": int(s.isna().sum()),
                      "distribución": ", ".join(f"{k}: {n}" for k, n in dist.items())
                      if s.nunique() <= 8 else f"min {s.min():.2f} · mediana {s.median():.2f} · max {s.max():.2f}"})
    return pd.DataFrame(filas).set_index("variable")

## 1. Normalización del texto

Antes de aplicar cualquier regla, cada respuesta se normaliza en cuatro pasos:

1. Descomposición Unicode **NFKD** y eliminación de los signos diacríticos.
2. Conversión a minúsculas.
3. Sustitución de saltos de línea y espacios repetidos por un solo espacio.
4. Eliminación de los espacios en los extremos.

Así, variantes como "Séptimo ", "septimo" y "SÉPTIMO" se comparan como el mismo texto. Para estandarizar duraciones, los números escritos con palabras ("una", "dos", "veinticuatro", "media") se convierten a dígitos. El siguiente ejemplo usa cadenas ilustrativas, no respuestas reales.

In [2]:
# Ejemplos ilustrativos de normalización y estandarización de horas (cadenas inventadas)
ejemplos = ["  Séptimo\nSEMESTRE ", "Una hora y media", "2-3 h", "Entre una y dos horas",
            "Como 7 horas o más", "menos de 1 hora", "10 min", "1 hora y 30 minutos"]
pd.DataFrame({"original": ejemplos,
              "normalizado": [normalizar(e) for e in ejemplos],
              "horas": [a_horas(e) for e in ejemplos]})

,original,normalizado,horas
0,Séptimo\nSEMESTRE,septimo semestre,NaN
1,Una hora y media,una hora y media,1.500000
2,2-3 h,2-3 h,2.500000
3,Entre una y dos horas,entre una y dos horas,1.500000
4,Como 7 horas o más,como 7 horas o mas,7.000000
5,menos de 1 hora,menos de 1 hora,0.500000
6,10 min,10 min,0.166667
7,1 hora y 30 minutos,1 hora y 30 minutos,1.500000


## 2. Seudonimización y corrección de la llave

El número de cuenta se reemplaza por un **seudónimo**: los primeros 12 caracteres del HMAC-SHA256 del número, calculado con una clave secreta. La clave se genera una sola vez y se guarda en `data/raw/`, que no se versiona. El seudónimo cumple tres propiedades:

- Es **determinista**: la misma cuenta produce el mismo identificador en los tres documentos, lo que permite la integración.
- Es **irreversible** sin la clave.
- No conserva la estructura del número de cuenta, como la generación de ingreso.

Antes de seudonimizar se atiende la cuenta de diez dígitos del Documento 1 detectada en la Fase 1. Si al borrar uno de sus dígitos se obtiene **exactamente una** cuenta del Documento 3 que no tiene pareja en el Documento 1, se considera un error de captura y se corrige. Si no hubiera coincidencia única, la cuenta se conservaría sin cambios.

In [3]:
# Corrección de la cuenta con longitud inválida (evidencia: coincidencia única con otro documento)
referencia = set(crudos[3]["cuenta"]) - set(crudos[1]["cuenta"])
crudos[1]["cuenta"], n_corregidas = corregir_cuenta_invalida(crudos[1]["cuenta"], referencia)
print("Cuentas corregidas en el Documento 1:", n_corregidas)
print("Cuentas con longitud distinta de 9 tras la corrección:",
      int((crudos[1]["cuenta"].astype(str).str.len() != 9).sum()))

clave = clave_seudonimo(RAW / ".clave_seudonimo")
limpios = {1: limpiar_doc1(crudos[1], clave), 2: limpiar_doc2(crudos[2], clave), 3: limpiar_doc3(crudos[3], clave)}
print("¿Queda alguna columna 'cuenta'?", any("cuenta" in df.columns for df in limpios.values()))
print("Seudónimos únicos por documento:", {d: df["id"].nunique() for d, df in limpios.items()})

Cuentas corregidas en el Documento 1: 1
Cuentas con longitud distinta de 9 tras la corrección: 0
¿Queda alguna columna 'cuenta'? False
Seudónimos únicos por documento: {1: 24, 2: 19, 3: 24}


## 3. Documento 1 — Música y estado de ánimo

### 3.1 Diagnóstico

In [4]:
# Diagnóstico del documento 1: forma, tipos, faltantes y duplicados
df = crudos[1]
print("Forma:", df.shape)
print("Tipos:", df.dtypes.astype(str).value_counts().to_dict())
print("Filas duplicadas (sin marca temporal):", int(df.drop(columns="marca_temporal").duplicated().sum()))
print("Cuentas duplicadas:", int(df["cuenta"].duplicated().sum()))
faltantes = df.isna().sum()
faltantes[faltantes > 0].rename("faltantes").to_frame()

Forma: (24, 18)
Tipos: {'str': 15, 'datetime64[us]': 1, 'Int64': 1, 'object': 1}
Filas duplicadas (sin marca temporal): 0
Cuentas duplicadas: 0


,faltantes
m_momentos,1


El documento tiene 24 registros y 18 variables, sin duplicados. El único faltante está en la pregunta sobre los momentos de escucha y corresponde al estudiante que declaró no escuchar música. No es una omisión aleatoria: la pregunta no le aplica.

### 3.2 Codificación

| Variable codificada | Origen | Regla principal |
|---------------------|--------|-----------------|
| `m_horas` | P2 | Duración en horas (sección 1). Los rangos se resumen en su punto medio y "x o más" se toma como x, la cota inferior |
| `m_horas_nivel` | `m_horas` | baja (< 2 h), media (2 a 4 h), alta (> 4 h), según `docs/negocio.md` §9.4 |
| `m_genero_grupo` | P1 | Primer género mencionado, agrupado en pop, rock/metal, electrónica, urbano (rap, reguetón, R&B, hip-hop), regional mexicano, variado y no escucha |
| `mejora_animo` | P5 | 1 si se describe una mejora o activación; 0 si se dice que el ánimo no cambia; faltante si solo se describe un estado ("bueno", "muy bueno") o la respuesta no trata del ánimo |
| `m_energia_cod` | P9 | 0 no influye, 1 poco o sin magnitud, 2 mucho |
| `m_concentracion_cod` | P10 | −1 distrae, 0 depende, 1 ayuda |
| `m_sueno_cod` | P11 | −1 perjudica, 0 no afecta o no la usa, 1 ayuda |
| `m_con_vs_sin_cod` | P12 | −1 mejor en silencio, 0 depende, 1 mejor con música |
| `m_maneja_emociones_cod` | P14 | 1 si la respuesta inicia con "sí" o "claro"; 0 si inicia con "no" |
| `m_importancia_cod` | P15 | 0 baja, 1 con reservas, 2 alta; las calificaciones x/10 se agrupan en < 5, 5 a 7 y ≥ 8 |
| `mom_*`, `plat_*`, `car_*`, `emo_*` | P3, P4, P13, P6 | Banderas 0/1 por mención de cada momento, plataforma, característica o género |

In [5]:
# Resultado de la codificación: valores codificados y faltantes por variable
resumen_codificacion(limpios[1], ["m_horas", "m_horas_nivel", "m_genero_grupo", "mejora_animo", "m_energia_cod", "m_concentracion_cod", "m_sueno_cod", "m_con_vs_sin_cod", "m_maneja_emociones_cod", "m_importancia_cod"])

,válidos,faltantes,distribución
variable,,,
m_horas,24,0,min 0.00 · mediana 4.00 · max 15.00
m_horas_nivel,24,0,"alta: 10, baja: 5, media: 9"
m_genero_grupo,24,0,"electronica: 3, no_escucha: 1, pop: 6, regiona..."
mejora_animo,20,4,"0.0: 3, 1.0: 17"
m_energia_cod,24,0,"0.0: 1, 1.0: 11, 2.0: 12"
m_concentracion_cod,23,1,"-1.0: 7, 0.0: 5, 1.0: 11"
m_sueno_cod,23,1,"-1.0: 2, 0.0: 14, 1.0: 7"
m_con_vs_sin_cod,21,3,"-1.0: 7, 0.0: 3, 1.0: 11"
m_maneja_emociones_cod,23,1,"0.0: 3, 1.0: 20"


In [6]:
# Banderas multirrespuesta: número de estudiantes que mencionan cada opción
banderas_d1 = list(MOMENTOS) + list(PLATAFORMAS) + list(CARACTERISTICAS) + [c for c in limpios[1] if c.startswith("emo_")]
limpios[1][banderas_d1].sum().astype(int).rename("menciones").to_frame().T

,mom_estudio,mom_traslado,mom_ejercicio,mom_descanso,mom_hogar,mom_trabajo,mom_todo_el_dia,plat_spotify,plat_youtube,plat_apple,plat_otra,car_ritmo,car_letra,car_volumen,car_instrumental,emo_pop,emo_rock_metal,emo_electronica,emo_urbano,emo_regional,emo_baladas,emo_clasica
menciones,10,18,9,4,4,4,3,11,13,2,3,19,13,3,1,13,9,2,6,4,5,1


**Tratamiento de faltantes.** Al estudiante que no escucha música se le asignó 0 en las banderas de momentos y plataformas, en lugar de dejarlas vacías: la ausencia de esos hábitos es un dato, no una omisión. Una respuesta de características ("no sabría decir") se dejó como faltante. Las respuestas que no pudieron codificarse sin interpretar de más también quedaron como faltantes.

**Hallazgo para el modelado.** La etiqueta `mejora_animo` tiene 17 casos positivos, 3 negativos y 4 faltantes. La clase minoritaria tiene menos de 4 casos, que es el mínimo fijado en la Fase 0 (`docs/negocio.md` §5). Por lo tanto, **no se cumple la condición para un modelo supervisado** sobre esta etiqueta, y la Fase 5 se centrará en el análisis no supervisado.

## 4. Documento 2 — Entorno universitario y bienestar

### 4.1 Diagnóstico

In [7]:
# Diagnóstico del documento 2: forma, tipos, faltantes y duplicados
df = crudos[2]
print("Forma:", df.shape)
print("Tipos:", df.dtypes.astype(str).value_counts().to_dict())
print("Filas duplicadas (sin marca temporal):", int(df.drop(columns="marca_temporal").duplicated().sum()))
print("Cuentas duplicadas:", int(df["cuenta"].duplicated().sum()))
faltantes = df.isna().sum()
faltantes[faltantes > 0].rename("faltantes").to_frame()

Forma: (19, 17)
Tipos: {'str': 16, 'Int64': 1}
Filas duplicadas (sin marca temporal): 0
Cuentas duplicadas: 0


,faltantes


El documento no tiene faltantes ni duplicados. Todas sus variables vienen como texto, incluidas la edad, el semestre y el tiempo de traslado.

### 4.2 Codificación

| Variable codificada | Origen | Regla principal |
|---------------------|--------|-----------------|
| `b_carrera_cod`, `b_semestre_cod`, `b_edad_cod` | P2 a P4 | Normalización a "MAC", a semestre numérico (incluye ordinales escritos) y a edad entera (incluye números escritos) |
| `b_traslado_min_cod` | P6 | Duración en minutos ("1 a 2 horas" → 90) |
| `b_desanimo_cod` | P10 | Likert 0 nunca, 1 rara vez, 2 a veces o 2 a 3 veces al mes, 3 frecuentemente o varias veces por semana, 4 muy frecuentemente, casi siempre o todos los días |
| `b_irritabilidad_cod` | P11 | Likert 0 a 4 sobre la facilidad para perder la paciencia (se respetan negaciones como "no pierdo la paciencia tan fácilmente") |
| `b_sueno_calidad_cod` | P8 | 0 mala, 1 algo mala o irregular, 2 regular, 3 buena |
| `b_concentracion_dif_cod` | P16 | 0 ninguna dificultad, 1 poca, 2 bastante, 3 mucha |
| `b_traslado_impacto_cod`, `b_apoyo_social_cod` | P13, P12 | Ordinales 0 a 3; las calificaciones x/10 se reescalan |
| `b_ambiente_hogar_cod`, `b_espacio_estudio_cod`, `b_interes_cod` | P7, P9, P14 | Ordinales 0 a 2; las respuestas con matiz ("mucha tranquilidad pero poco espacio") reciben el nivel intermedio |
| `b_alimentacion_cod` | P15 | 1 si se reporta un cambio; la negación solo cuenta si abre la respuesta |
| `iep`, `iep_nivel` | P10, P11 | Índice de estrés percibido: promedio de desánimo e irritabilidad (0 a 4); bajo < 1.5, medio 1.5 a 2.5, alto > 2.5 |

In [8]:
# Resultado de la codificación: valores codificados y faltantes por variable
resumen_codificacion(limpios[2], ["b_carrera_cod", "b_semestre_cod", "b_edad_cod", "b_traslado_min_cod", "b_desanimo_cod", "b_irritabilidad_cod", "iep", "iep_nivel", "b_sueno_calidad_cod", "b_concentracion_dif_cod", "b_traslado_impacto_cod", "b_apoyo_social_cod", "b_ambiente_hogar_cod", "b_espacio_estudio_cod", "b_interes_cod", "b_alimentacion_cod", "b_convivencia_cod"])

,válidos,faltantes,distribución
variable,,,
b_carrera_cod,18,1,MAC: 18
b_semestre_cod,19,0,7.0: 19
b_edad_cod,19,0,"21.0: 8, 22.0: 5, 23.0: 2, 24.0: 3, 25.0: 1"
b_traslado_min_cod,19,0,min 10.00 · mediana 60.00 · max 360.00
b_desanimo_cod,18,1,"1.0: 5, 2.0: 3, 3.0: 3, 4.0: 7"
b_irritabilidad_cod,19,0,"1.0: 11, 2.0: 2, 3.0: 1, 4.0: 5"
iep,18,1,"1.0: 4, 1.5: 1, 2.0: 3, 2.5: 5, 3.0: 1, 3.5: 1..."
iep_nivel,18,1,"alto: 5, bajo: 4, medio: 9"
b_sueno_calidad_cod,19,0,"0.0: 5, 1.0: 6, 2.0: 5, 3.0: 3"


**Faltantes generados por la codificación:**

- Una respuesta de carrera ("UNAM") no indica la carrera. Se deja como faltante y el registro se conserva, porque el resto de sus respuestas sí son válidas.
- Una respuesta de desánimo ("5 veces") no indica el periodo de referencia, así que no puede ubicarse en la escala. Por la misma razón, ese estudiante no tiene IEP.
- Dos respuestas de interés no tienen un nivel interpretable: "se sigue manteniendo estable" y un "4" sin escala.

Ninguno de estos valores se imputó. El IEP se calcula para 18 de los 19 estudiantes.

## 5. Documento 3 — Desempeño académico por el uso de IA

### 5.1 Diagnóstico

In [9]:
# Diagnóstico del documento 3: forma, tipos, faltantes y duplicados
df = crudos[3]
print("Forma:", df.shape)
print("Tipos:", df.dtypes.astype(str).value_counts().to_dict())
print("Filas duplicadas (sin marca temporal):", int(df.drop(columns="marca_temporal").duplicated().sum()))
print("Cuentas duplicadas:", int(df["cuenta"].duplicated().sum()))
faltantes = df.isna().sum()
faltantes[faltantes > 0].rename("faltantes").to_frame()

Forma: (24, 18)
Tipos: {'str': 17, 'Int64': 1}
Filas duplicadas (sin marca temporal): 0
Cuentas duplicadas: 0


,faltantes
ia_calificaciones,1
ia_pensamiento_critico,1


El documento tiene dos faltantes de origen: una pregunta sobre calificaciones y otra sobre pensamiento crítico, ambas del mismo estudiante. Se conservan como faltantes porque no hay información para completarlos.

### 5.2 Codificación

| Variable codificada | Origen | Regla principal |
|---------------------|--------|-----------------|
| `ia_horas` | P12 | Duración en horas ("5 minutos" → 0.08; "menos de 1" → 0.5) |
| `ia_semestre_inicio_cod` | P13 | Semestre numérico; "hace k semestres" → 7 − k |
| `ia_guia_vs_atajo_cod` | P15 | guía, atajo o ambas; "principalmente como guía" se codifica como guía |
| `ia_estres_cod` | P9 | −1 aumenta el estrés, 0 sin efecto, 1 lo disminuye |
| `ia_independencia_cod` | P8 | −1 debilita, 0 depende o ambas, 1 fortalece; las respuestas condicionales ("debilita si…") se codifican como 0 |
| `ia_impacto_tiempo_cod`, `ia_calificaciones_cod`, `ia_pensamiento_critico_cod` | P6, P7, P11 | Escalas −1, 0, 1 según la dirección del efecto descrito |
| `ia_reemplazo_laboral_cod` | P14 | sí, no o parcial |
| `ia_motivo_cod` | P16 | comprensión, tiempo, ambos u otro |
| `ia_otros_usos_cod` | P17 | 1 si menciona otros usos |
| `ia_*` (herramientas), `fase_*` | P2, P3 | Banderas por herramienta y por fase de uso; `ia_n_herramientas` es la suma de herramientas |

In [10]:
# Resultado de la codificación: valores codificados y faltantes por variable
resumen_codificacion(limpios[3], ["ia_horas", "ia_semestre_inicio_cod", "ia_guia_vs_atajo_cod", "ia_estres_cod", "ia_independencia_cod", "ia_impacto_tiempo_cod", "ia_calificaciones_cod", "ia_pensamiento_critico_cod", "ia_reemplazo_laboral_cod", "ia_motivo_cod", "ia_otros_usos_cod", "ia_n_herramientas"])

,válidos,faltantes,distribución
variable,,,
ia_horas,24,0,min 0.08 · mediana 1.50 · max 8.00
ia_semestre_inicio_cod,24,0,"1.0: 3, 2.0: 3, 3.0: 8, 4.0: 7, 5.0: 2, 6.0: 1"
ia_guia_vs_atajo_cod,23,1,"ambas: 13, atajo: 2, guia: 8"
ia_estres_cod,24,0,"-1.0: 1, 0.0: 3, 1.0: 20"
ia_independencia_cod,21,3,"-1.0: 10, 0.0: 9, 1.0: 2"
ia_impacto_tiempo_cod,21,3,"0.0: 2, 1.0: 19"
ia_calificaciones_cod,19,5,"-1.0: 1, 0.0: 9, 1.0: 9"
ia_pensamiento_critico_cod,22,2,"-1.0: 8, 0.0: 12, 1.0: 2"
ia_reemplazo_laboral_cod,23,1,"no: 11, parcial: 4, si: 8"


In [11]:
# Banderas de herramientas y fases de uso
limpios[3][list(HERRAMIENTAS) + list(FASES)].sum().astype(int).rename("menciones").to_frame().T

,ia_chatgpt,ia_gemini,ia_claude,ia_deepseek,ia_copilot,ia_notebooklm,ia_qwen,fase_investigacion,fase_estructuracion,fase_redaccion,fase_correccion,fase_repaso
menciones,11,15,8,3,2,3,1,16,6,8,11,6


La variable `ia_estres_cod` está muy concentrada: la gran mayoría de los estudiantes reporta que la IA *disminuye* su estrés. Esto limita su capacidad para distinguir grupos en la Fase 4. En `ia_guia_vs_atajo_cod`, la categoría "atajo" tiene solo 2 casos, así que para los indicadores se agrupará con "ambas", conforme a la regla de mínimo 3 casos de `docs/negocio.md` §9.4.

## 6. Valores atípicos

Los atípicos se detectan con el criterio del rango intercuartílico (fuera de Q1 − 1.5·IQR o de Q3 + 1.5·IQR) en las variables cuantitativas. **Se marcan pero no se eliminan.** Son auto-reportes plausibles para algunas personas, por ejemplo música de fondo durante toda la jornada o traslados que incluyen trabajo. Para reducir su influencia, los indicadores usarán la mediana además de la media, y en la Fase 4 se revisará si cambian las conclusiones.

In [12]:
# Tabla de atípicos por variable cuantitativa (solo estadísticos agregados)
filas = []
for d, var, marca in [(1, "m_horas", "m_horas_atipico"), (2, "b_edad_cod", "b_edad_atipico"),
                      (2, "b_traslado_min_cod", "b_traslado_min_atipico"), (3, "ia_horas", "ia_horas_atipico")]:
    s, a = limpios[d][var], limpios[d][marca]
    q1, q3 = s.quantile([.25, .75])
    filas.append({"doc": d, "variable": var, "n": int(s.notna().sum()), "mediana": s.median(),
                  "Q1": q1, "Q3": q3, "límite_superior": q3 + 1.5 * (q3 - q1),
                  "atípicos": int(a.sum()), "valores_atípicos": sorted(s[a == 1].round(2).tolist())})
pd.DataFrame(filas).set_index(["doc", "variable"])

n  mediana      Q1    Q3  límite_superior  atípicos  \
doc variable                                                                   
1   m_horas             24      4.0   2.375   5.5          10.1875         1   
2   b_edad_cod          19     22.0  21.000  23.0          26.0000         0   
    b_traslado_min_cod  19     60.0  35.000  75.0         135.0000         1   
3   ia_horas            24      1.5   1.000   2.0           3.5000         3   

                       valores_atípicos  
doc variable                             
1   m_horas                      [15.0]  
2   b_edad_cod                       []  
    b_traslado_min_cod          [360.0]  
3   ia_horas            [4.0, 6.0, 8.0]

## 7. Respuestas copiadas

Se contó, para cada estudiante, cuántas de sus respuestas abiertas de al menos cinco palabras son idénticas a la respuesta de otro estudiante. Las respuestas cortas ("El ritmo", "Muy fácilmente") se excluyen porque coinciden de forma natural. Varias coincidencias largas en un mismo registro sugieren respuestas copiadas o duplicadas. Como no puede demostrarse, el registro **se conserva y se marca**, y en la Fase 4 se hará un análisis de sensibilidad excluyéndolo.

In [13]:
# Distribución del número de respuestas largas idénticas a las de otro estudiante
pd.DataFrame({f"Doc {d}": limpios[d][f"copias_d{d}"].value_counts().sort_index() for d in (1, 2, 3)}) \
  .fillna(0).astype(int).rename_axis("respuestas_copiadas")

,Doc 1,Doc 2,Doc 3
respuestas_copiadas,,,
0,17,19,20
1,6,0,2
2,1,0,1
4,0,0,1


In [14]:
# Registros con 3 o más respuestas copiadas en algún documento (se identifican por seudónimo, no por cuenta)
sospechosos = {d: set(limpios[d].loc[limpios[d][f"copias_d{d}"] >= 3, "id"]) for d in (1, 3)}
todos = set().union(*sospechosos.values())
print("Estudiantes marcados:", len(todos))
print("¿Aparecen en el Documento 2 (núcleo integrable)?", len(todos & set(limpios[2]["id"])))
for d in (1, 3):
    limpios[d]["sospecha_copia"] = limpios[d]["id"].isin(todos).astype(int)

Estudiantes marcados: 1
¿Aparecen en el Documento 2 (núcleo integrable)? 0


## 8. Guardado de los conjuntos limpios

Cada documento se guarda en `data/processed/` con el sufijo `_clean`. Los archivos contienen el seudónimo, las respuestas originales en texto (para el análisis descriptivo de texto), las variables codificadas y las banderas de calidad. No contienen números de cuenta. También se exporta a `docs/variables_codificadas.csv` un resumen de las variables codificadas, sin datos individuales.

In [15]:
# Guardado de los conjuntos limpios y del resumen de variables codificadas
nombres = {1: "doc1_musica_clean.csv", 2: "doc2_bienestar_clean.csv", 3: "doc3_ia_clean.csv"}
for d, df in limpios.items():
    assert "cuenta" not in df.columns
    df.to_csv(PROCESSED / nombres[d], index=False, encoding="utf-8")

originales = {c for d in (1, 2, 3) for c in crudos[d].columns}
resumen = []
for d, df in limpios.items():
    for c in df.columns:
        if c in originales or c == "id":
            continue
        resumen.append({"doc": d, "variable": c, "tipo": str(df[c].dtype), "válidos": int(df[c].notna().sum()),
                        "faltantes": int(df[c].isna().sum()), "valores_distintos": int(df[c].nunique())})
variables_cod = pd.DataFrame(resumen)
variables_cod.to_csv(RAIZ / "docs" / "variables_codificadas.csv", index=False, encoding="utf-8-sig")
print({nombres[d]: limpios[d].shape for d in limpios})
print("Variables derivadas:", len(variables_cod))
variables_cod.groupby("doc")[["válidos", "faltantes"]].sum()

{'doc1_musica_clean.csv': (24, 53), 'doc2_bienestar_clean.csv': (19, 37), 'doc3_ia_clean.csv': (24, 45)}
Variables derivadas: 82


,válidos,faltantes
doc,,
1,818,22
2,374,6
3,633,15


## 9. Bitácora de decisiones

| # | Documento | Decisión | Justificación |
|---|-----------|----------|---------------|
| 1 | 1 | Se corrigió una cuenta de 10 dígitos borrando un dígito | Coincide de forma única con la cuenta del Documento 3 que no tenía pareja. Sin coincidencia única no se habría modificado |
| 2 | Todos | El número de cuenta se sustituyó por un seudónimo HMAC-SHA256 con clave secreta fuera de Git | Permite unir los documentos sin exponer datos personales |
| 3 | Todos | Texto normalizado con NFKD, minúsculas y espacios unificados | Elimina variantes de escritura antes de aplicar las reglas |
| 4 | 1, 3 | Horas: rango → punto medio; "x o más" → x; "menos de 1" → 0.5; minutos → fracción de hora | Criterio uniforme y conservador; "o más" usa la cota inferior para no inflar el valor |
| 5 | 2 | Traslado en minutos; "1 a 2 horas" → 90 | Mismo criterio de punto medio |
| 6 | 2 | Desánimo e irritabilidad en una escala Likert de 0 a 4, con frecuencias explícitas ubicadas por periodo | Hace comparables las respuestas verbales y numéricas; si no hay periodo ("5 veces") queda como faltante |
| 7 | Todos | Las respuestas sin magnitud o dirección interpretable quedan como faltantes | Principio de no inventar datos |
| 8 | 2 | Las calificaciones x/10 se reescalan a la escala ordinal de la variable | Usa la información numérica que dio el estudiante |
| 9 | 1 | En `mejora_animo`, describir solo un estado ("bueno") no cuenta como mejora | La pregunta pide un antes y un después |
| 10 | 1 | El género principal es el primero mencionado; "poprock" se agrupa en pop | El orden refleja la preferencia declarada; regla simple y reproducible |
| 11 | 1 | Quien no escucha música recibe 0 en momentos y plataformas | Es ausencia del hábito, no falta de respuesta |
| 12 | 2 | La carrera "UNAM" queda como faltante y el registro se conserva | No especifica la carrera, pero el resto de las respuestas es válido |
| 13 | 1, 2, 3 | Los atípicos (IQR) se marcan y no se eliminan | Son auto-reportes plausibles; se usarán estadísticos robustos |
| 14 | 1, 3 | Las respuestas largas idénticas entre estudiantes se marcan (`sospecha_copia`) y no se eliminan | No hay prueba de invalidez; se hará un análisis de sensibilidad en la Fase 4 |
| 15 | Todos | No se imputa ningún valor | Con n = 19, cualquier imputación tendría un peso desproporcionado en los resultados |

## 10. Conclusiones de la fase

- Los tres documentos quedaron depurados y guardados por separado en `data/processed/` (`doc1_musica_clean.csv`, `doc2_bienestar_clean.csv` y `doc3_ia_clean.csv`), sin números de cuenta.
- Todas las variables que operacionalizan las hipótesis H1 a H4 tienen ahora una versión numérica o categórica. Los faltantes generados por la codificación son pocos y están justificados.
- El IEP está disponible para 18 estudiantes del Documento 2.
- La etiqueta de mejora de ánimo está muy desbalanceada (17 frente a 3), así que el modelado supervisado no es viable con este criterio. Esta decisión se formalizará en la Fase 5.